In [ ]:
!git clone https://github.com/cfiltnlp/MUStARD_Plus_Plus.git

Cloning into 'MUStARD_Plus_Plus'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 41 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 232.82 KiB | 609.00 KiB/s, done.
Resolving deltas: 100% (11/11), done.


In [ ]:
!mkdir data
!mv MUStARD_Plus_Plus mustard++
!mv mustard++/ data/

In [ ]:
!cp /content/drive/MyDrive/final_utterance_videos.zip /content/data/mustard++/

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
import json
import pandas as pd
from faster_whisper import WhisperModel
from jiwer import wer
import moviepy as mp
from tqdm.auto import tqdm

In [6]:
!pip install faster_whisper jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 119.5 MB/s eta 0:00:00


In [10]:
!pip install moviepy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 4.6 MB/s eta 0:00:00


# Audio Preprocessing

Collecting timestaps and probabilities of target utterance to crop audio while training

In [ ]:
from moviepy.video.io.VideoFileClip import VideoFileClip

In [ ]:
class UtteranceMetadataProcessor:
    """process dataset utterances to extract timestamps and analyse asr quality"""

    def __init__(self, model_size, device, compute_type):
        """
        initialise whisper model and configs
        """

        # load whisper model
        self.model = WhisperModel(model_size, device=device, compute_type=compute_type)

        # define computation parameters
        self.device = device

    @staticmethod
    def extract_audio_from_video(video_path, audio_output_path) -> None:
        """
        Convert video segment to audio for whisper processing
        :param video_path: path to video file
        :param audio_output_path: path to saved audio file
        :return: None
        """

        # load video file
        video = VideoFileClip(video_path)

        # extract audio track
        video.audio.write_audiofile(audio_output_path, logger=None)

        # close video instance
        video.close()

    @staticmethod
    def calculate_error_metrics(reference_text, hypothesis_text) -> float:
        """
        Compute word error rate between csv text and whisper output
        :param reference_text: target correct sentence
        :param hypothesis_text: text to compare
        :return: WER-score
        """
        # return zero if texts are identical or empty
        if not reference_text or not hypothesis_text:
            return 0.0

        # calculate normalized error rate
        return wer(reference_text.lower(), hypothesis_text.lower())

    def process_utterances(self, csv_path, video_dir, output_json_path) -> None:
        """
        Main loop to process every utterance and save statistics
        :param csv_path: path to csv file with transcriptions
        :param video_dir: path to directory with videos
        :param output_json_path: path to output JSON file
        :return: None
        """
        # read dataset metadata
        data_frame = pd.read_csv(csv_path)

        # filter only target utterances
        utterance_data = data_frame[data_frame['KEY'].str.endswith('_u')]

        results = []

        processed_count = 0

        # iterate through each utterance row
        for index, row in tqdm(utterance_data.iterrows(), position=0, leave=True, desc="Iter transcriptions"):
            scene_key = row['KEY']
            ground_truth_text = row['SENTENCE']
            video_filename = f"{scene_key}.mp4"
            video_path = os.path.join(video_dir, video_filename)

            # check if video file exists
            if not os.path.exists(video_path):
                continue

            temp_audio = "temp_audio.wav"

            # prepare audio for whisper
            self.extract_audio_from_video(video_path, temp_audio)

            # transcribe audio with word level timestamps
            segments, info = self.model.transcribe(temp_audio, word_timestamps=True)
            segments = list(segments)

            # aggregate transcription results
            full_transcription = ""
            word_metadata = []

            first_speech_timestamp = None
            last_speech_timestamp = 0.0

            # extract detailed segment information
            for segment in segments:
                full_transcription += segment.text + " "
                for word in segment.words:
                    word_metadata.append({
                        "word": word.word,
                        "start": word.start,
                        "end": word.end,
                        "probability": word.probability
                    })

                    # capture the very first moment of speech
                    if first_speech_timestamp is None:
                        first_speech_timestamp = word.start

                    # update last known speech moment
                    last_speech_timestamp = max(last_speech_timestamp, word.end)

            # handle cases with no detected speech
            if first_speech_timestamp is None:
                first_speech_timestamp = 0.0

            # clean transcription text
            full_transcription = full_transcription.strip()

            # analyse asr performance
            error_rate = self.calculate_error_metrics(ground_truth_text, full_transcription)

            # update iteration counter
            processed_count += 1

            # display asr metrics for verification
            # log only initial examples to verify quality
            if processed_count <= 10:
                # log processing status
                print(f"processing item: {scene_key}")
                print(f"reference: {ground_truth_text}")
                print(f"whisper:   {full_transcription}")
                print(f"wer score: {error_rate:.4f}")
                print(f"speech window: {first_speech_timestamp:.2f}s - {last_speech_timestamp:.2f}s")
                print("-" * 30)

            # store all metadata including sarcasm labels
            results.append({
                "key": scene_key,
                "speaker": row['SPEAKER'],
                "show": row['SHOW'],
                "sarcasm_label": row['Sarcasm'],
                "reference_text": ground_truth_text,
                "whisper_text": full_transcription,
                "word_error_rate": error_rate,
                "start_of_speech": first_speech_timestamp,
                "end_of_speech": last_speech_timestamp,
                "detailed_words": word_metadata
            })

            # remove temporary file
            if os.path.exists(temp_audio):
                os.remove(temp_audio)

        # save statistics to local storage
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=4)


In [ ]:
def process_masterpp_dataset(path_output: str, device: str = "cpu") -> None:
    """
    Process master++ dataset to get transcriptions and measure WER score
    :param path_output: output path of directory for JSON file with statistics
    :param device: device to use for model
    :return: None
    """
    # configure paths and parameters
    input_csv = "/content/data/mustard++/mustard++_text.csv"
    input_videos = "/content/data/mustard++/all_videos/final_utterance_videos"
    output_data = os.path.join(path_output, "utterance_asr_stats.json")

    # initialize processor with gpu settings
    processor = UtteranceMetadataProcessor(
        model_size="large-v3",
        device=device,
        compute_type="float16"
    )

    # execute batch processing
    processor.process_utterances(input_csv, input_videos, output_data)


In [ ]:
process_masterpp_dataset(path_output="/content/data/mustard++/", device="cuda")

Iter transcriptions: 0it [00:00, ?it/s]

processing item: 1_10004_u
reference: And of those few months, how long have you been a demented sex pervert?
whisper:   And of those few months, how long have you been a demented sex pervert?
wer score: 0.0000
speech window: 0.64s - 4.90s
------------------------------
processing item: 1_10009_u
reference: Let the dead man talk. So, why do you think that?
whisper:   Let the dead man talk.  So why do you think that?  Uh, well.
wer score: 0.2727
speech window: 0.00s - 4.82s
------------------------------
processing item: 1_1001_u
reference: What else? Sell it on eBay as "slightly used."
whisper:   What else?  Selling on eBay is slightly used.
wer score: 0.5556
speech window: 0.00s - 2.04s
------------------------------
processing item: 1_1003_u
reference: Good idea, sit with her. Hold her, comfort her. And if the moment feels right, see if you can cop a feel.
whisper:   to her again good idea sit with her hold her comfort her and if the moment feels right  see if you can cop a feel
wer 

index 99344 is out of bounds for axis 0 with size 99344
  warnings.warn("Error in file %s, "%(self.filename)+



In [ ]:
!pip install torch torchaudio moviepy scikit-learn transformers thop

# Train model

### Extract Audio

In [ ]:
import os
from moviepy.video.io.VideoFileClip import VideoFileClip

In [12]:
class AudioExtractor:
    """utility to extract audio tracks from video files for faster training"""

    def __init__(self, video_dir, audio_dir, target_sr=16000):
        """initialize paths and audio settings"""
        self.video_dir = video_dir
        self.audio_dir = audio_dir
        self.target_sr = target_sr

        # create output directory if it does not exist
        if not os.path.exists(self.audio_dir):
            os.makedirs(self.audio_dir)

    def run(self):
        """iterate through videos and save audio as wav files"""
        video_files = [f for f in os.listdir(self.video_dir) if f.endswith('.mp4')]

        print(f"found {len(video_files)} videos. starting extraction...")

        for filename in video_files:
            video_path = os.path.join(self.video_dir, filename)
            # define output name by replacing extension
            audio_name = filename.replace('.mp4', '.wav')
            audio_path = os.path.join(self.audio_dir, audio_name)

            # skip if already extracted to save time
            if os.path.exists(audio_path):
                continue

            try:
                # load video and extract audio channel
                clip = VideoFileClip(video_path)
                # write audio with specific sampling rate
                clip.audio.write_audiofile(
                    audio_path,
                    fps=self.target_sr,
                    nbytes=2,
                    codec='pcm_s16le',
                    verbose=False,
                    logger=None
                )
                clip.close()
            except Exception as e:
                print(f"error processing {filename} error details {e}")

        print("extraction finished success")

## Dataset and Model classes

In [13]:
import time
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torchaudio
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import WhisperModel, WhisperFeatureExtractor

In [14]:
# set seeds for reproducibility
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)

In [15]:
import gc
from tqdm.auto import tqdm

In [16]:
class MustardAudioDataset(Dataset):
    """custom dataset for audio processing with mono conversion and memory efficiency"""

    def __init__(self, data_frame, stats_json, audio_dir, feature_extractor, augment=False):
        """initialize and load essential timestamps only"""
        self.df = data_frame
        self.audio_dir = audio_dir
        self.feature_extractor = feature_extractor
        self.augment = augment
        self.target_sr = 16000

        with open(stats_json, 'r', encoding='utf-8') as f:
            full_stats = json.load(f)

        self.timestamps = {
            item['key']: {
                'start': item['start_of_speech'],
                'end': item['end_of_speech']
            } for item in full_stats
        }

        del full_stats
        gc.collect()

    def __len__(self):
        return len(self.df)

    def apply_augmentations(self, waveform):
        """apply audio augmentations"""
        pitch_factor = random.uniform(-2.0, 2.0)
        waveform = torchaudio.transforms.PitchShift(self.target_sr, pitch_factor)(waveform)

        stretch_factor = random.uniform(0.8, 1.2)
        new_freq = int(self.target_sr * stretch_factor)
        waveform = torchaudio.transforms.Resample(new_freq, self.target_sr)(waveform)
        return waveform

    def __getitem__(self, idx):
        """load audio slice, convert to mono and extract features"""
        row = self.df.iloc[idx]
        key = row['KEY']
        label = row['label']

        audio_path = f"{self.audio_dir}/{key}.wav"
        waveform, sample_rate = torchaudio.load(audio_path)

        # FIXED: convert stereo to mono to avoid dimension mismatch in conv1d
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # normalize sampling rate
        if sample_rate != self.target_sr:
            waveform = torchaudio.transforms.Resample(sample_rate, self.target_sr)(waveform)

        # extract boundaries
        ts = self.timestamps.get(key)
        start_sample = int(ts['start'] * self.target_sr)
        end_sample = int(ts['end'] * self.target_sr)

        # crop to speech window
        waveform = waveform[:, start_sample:end_sample]

        # handle empty crop safety
        if waveform.shape[1] < 160: # less than 10ms
            waveform = torch.zeros((1, 16000)) # fallback to 1s silence

        if self.augment:
            waveform = self.apply_augmentations(waveform)

        # extract whisper features (must be mono numpy array)
        features = self.feature_extractor(
            waveform.squeeze(0).detach().cpu().numpy(),
            sampling_rate=self.target_sr,
            return_tensors="pt"
        ).input_features.squeeze(0)

        return {
            "input_features": features,
            "label": torch.tensor(label, dtype=torch.long),
            "key": key
        }

In [17]:
class CachedMustardDataset(Dataset):
    """lightning fast dataset with on-the-fly specaugment for cached tensors"""

    def __init__(self, data_frame, feature_dir, *args, augment=False, **kwargs):
        """initialize and ignore unused arguments from raw dataset signature"""
        self.df = data_frame
        # feature_dir is where your .pt files are stored
        self.feature_dir = feature_dir
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def apply_spec_augment(self, mel_spec):
        """apply frequency and time masking to the mel spectrogram"""
        # frequency masking to hide pitch details
        freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
        # time masking to hide temporal details
        time_mask = torchaudio.transforms.TimeMasking(time_mask_param=35)

        augmented_mel = freq_mask(mel_spec)
        augmented_mel = time_mask(augmented_mel)
        return augmented_mel

    def __getitem__(self, idx):
        """load pre-calculated tensor and apply gpu-based augmentation"""
        row = self.df.iloc[idx]
        key = row['KEY']

        # load pre-saved tensor from disk
        feature_path = os.path.join(self.feature_dir, f"{key}.pt")
        features = torch.load(feature_path)

        if self.augment:
            features = self.apply_spec_augment(features)

        return {
            "input_features": features,
            "label": torch.tensor(row['label'], dtype=torch.long),
            "key": key
        }

In [18]:
class RobustAttentiveClassifier(nn.Module):
    """audio classifier with multihead attention pooling and normalization"""

    def __init__(self, backbone, hidden_size, num_heads=4):
        """setup architecture with frozen encoder and trainable head"""
        super().__init__()
        self.backbone = backbone
        # freeze encoder parameters to save resources
        for param in self.backbone.parameters():
            param.requires_grad = False

        self.layer_norm = nn.LayerNorm(hidden_size)
        self.mha = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
        # learnable query for pooling axis
        self.query = nn.Parameter(torch.randn(1, 1, hidden_size))

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 2)
        )

    def forward(self, input_features):
        """process mel features through backbone and attention"""
        # whisper encoder returns hidden states directly
        hidden_states = self.backbone(input_features).last_hidden_state
        hidden_states = self.layer_norm(hidden_states)

        # attentive pooling
        batch_size = hidden_states.size(0)
        query = self.query.expand(batch_size, -1, -1)
        attn_out, _ = self.mha(query, hidden_states, hidden_states)

        pooled = attn_out.squeeze(1)
        logits = self.head(pooled)
        return logits, pooled


## Profiler for model cost metric

In [19]:
!pip install thop

In [20]:
from thop import profile

class PerformanceProfiler:
    """monitor and calculate computational cost and quality metrics"""

    def __init__(self, device):
        """initialize profiler with target device"""
        self.device = device

    def measure_flops(self, model, sample_input):
        """calculate floating point operations per second using thop"""
        model.eval()
        # wrap input for thop profiling
        flops, params = profile(model, inputs=(sample_input.to(self.device),), verbose=False)
        return flops, params

    def measure_latency(self, model, dataloader, runs=100):
        """calculate average inference time per sample in milliseconds"""
        model.eval()
        latencies = []

        # perform warmup to stabilize gpu clocks
        with torch.no_grad():
            batch = next(iter(dataloader))
            warmup_input = batch["input_features"].to(self.device)
            for _ in range(10):
                _ = model(warmup_input)

        # measurement loop
        with torch.no_grad():
            for i, batch in enumerate(dataloader):
                if i >= runs:
                    break
                inputs = batch["input_features"].to(self.device)

                if self.device == "cuda":
                    torch.cuda.synchronize()

                start_time = time.perf_counter()
                _ = model(inputs)

                if self.device == "cuda":
                    torch.cuda.synchronize()
                end_time = time.perf_counter()

                # normalize latency by batch size
                latencies.append((end_time - start_time) * 1000 / inputs.size(0))

        return np.mean(latencies), np.std(latencies)

    def get_memory_stats(self):
        """return peak vram consumption in megabytes"""
        if self.device == "cuda":
            return torch.cuda.max_memory_allocated() / (1024 * 1024)
        return 0.0


# Whisper Small Encoder

## Train and evaluate

In [ ]:
def train_and_evaluate(train_df, val_df, test_df, stats_json, audio_dir, cached=False, feature_dir=None):
    """complete execution flow for training and metric collection"""

    # device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # load components for whisper small
    model_id = "openai/whisper-small"
    feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
    backbone = WhisperModel.from_pretrained(model_id).get_encoder()

    # initialize custom classes provided earlier
    if cached:
      train_dataset = CachedMustardDataset(train_df, feature_dir, augment=True)
      val_dataset = CachedMustardDataset(val_df, feature_dir, augment=False)
      test_dataset = CachedMustardDataset(test_df, feature_dir, augment=False)
    else:
      train_dataset = MustardAudioDataset(train_df, stats_json, audio_dir, feature_extractor, augment=True)
      val_dataset = MustardAudioDataset(val_df, stats_json, audio_dir, feature_extractor, augment=False)
      test_dataset = MustardAudioDataset(test_df, stats_json, audio_dir, feature_extractor, augment=False)


    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    # clear everything before starting
    gc.collect()
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
      torch.cuda.reset_peak_memory_stats()

    # init model with whisper hidden size
    model = RobustAttentiveClassifier(backbone, hidden_size=768)
    model.to(device)

    # simple training setup
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    # train loop
    best_val_f1 = 0.0
    for epoch in tqdm(range(30), position=0, leave=True, desc="Processing epoch"):
        model.train()
        for batch in tqdm(train_loader, position=1, leave=True, desc="Processing batches"):
            x = batch["input_features"].to(device)
            y = batch["label"].to(device)

            optimizer.zero_grad()
            logits, _ = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        # run validation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                x = batch["input_features"].to(device)
                y = batch["label"].to(device)
                logits, _ = model(x)
                all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                all_labels.extend(y.cpu().numpy())

        current_f1 = f1_score(all_labels, all_preds, average='macro')
        print(f"epoch {epoch} validation fone score {current_f1:.4f}")

        if current_f1 > best_val_f1:
            best_val_f1 = current_f1
            torch.save(model.state_dict(), "best_audio_model.pt")

    # final performance evaluation
    model.load_state_dict(torch.load("best_audio_model.pt"))
    model.eval()

    profiler = PerformanceProfiler(device)

    # post training feature extraction for next fusion models
    model.eval()
    embeddings_dict = {}
    with torch.no_grad():
        for batch in test_loader:
            x = batch["input_features"].to(device)
            key = batch["key"][0]
            _, pooled = model(x)
            embeddings_dict[key] = pooled.cpu().numpy()

    # save as numpy object file
    np.save("audio_embeddings.npy", embeddings_dict)

    # quality metrics on test set
    test_preds, test_labels, embeddings_dict = [], [], {}
    with torch.no_grad():
        for batch in test_loader:
            x = batch["input_features"].to(device)
            y = batch["label"].to(device)
            key = batch["key"][0]

            logits, pooled = model(x)

            test_preds.append(torch.argmax(logits, dim=1).item())
            test_labels.append(y.item())
            # store pooled output for track three fusion
            embeddings_dict[key] = pooled.cpu().numpy().tolist()

    # compute final scores
    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds, average='macro')

    # computational metrics
    sample_batch = next(iter(test_loader))["input_features"]
    flops, params = profiler.measure_flops(model, sample_batch)
    mean_lat, std_lat = profiler.measure_latency(model, test_loader)
    vram = profiler.get_memory_stats()

    # aggregate all research data
    final_report = {
        "test_accuracy": acc,
        "test_f1_macro": f1,
        "inference_latency_ms": mean_lat,
        "latency_std_ms": std_lat,
        "total_flops": flops,
        "total_gflops": flops / 1e9,
        "model_parameters_count": params,
        "peak_vram_mb": vram
    }

    print("\nfinal research report")
    print(json.dumps(final_report, indent=4))

    # save embeddings for multimodal track
    with open("audio_embeddings.json", "w") as f:
        json.dump(embeddings_dict, f)

    return final_report, model

In [ ]:
def run_audio_classification_pipeline(csv_path, video_dir, stats_json, extract=True):
    """main execution flow for audio only research track"""

    # data preparation
    df_raw = pd.read_csv(csv_path)

    # filter only utterance rows
    df = df_raw[df_raw["KEY"].str.endswith("_u")].copy()

    df["label"] = df["Sarcasm"].astype(int)

    # eplicate split from track one using seed 42
    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        random_state=42,
        stratify=df["label"]
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        stratify=temp_df["label"]
    )

    print(f"data split done train {len(train_df)} val {len(val_df)} test {len(test_df)}")

    # extract audio tracks from video files
    # create audio directory for wav storage
    audio_dir = "mustardpp_audio_wav"
    if extract:
      extractor = AudioExtractor(video_dir, audio_dir)
      extractor.run()

    # train and evaluate model
    # this function contains our research metrics and profiling
    # we use whisper small as the initial backbone
    report, model = train_and_evaluate(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        stats_json=stats_json,
        audio_dir=audio_dir
    )

    # save final report to json
    with open("audio_track_report.json", "w") as f:
        json.dump(report, f, indent=4)

    print("pipeline execution finished check audio_embeddings.npy for track three")
    return model

In [21]:
CSV_PATH = "/content/mustard++_text.csv"
VIDEO_DIR = "/content/final_utterance_videos"
STATS_JSON = "/content/utterance_asr_stats.json"
AUDIO_DIR = "/content/mustardpp_audio_wav"

In [ ]:

whisper_model = run_audio_classification_pipeline(CSV_PATH, VIDEO_DIR, STATS_JSON)


data split done train 840 val 180 test 181
found 1203 videos. starting extraction...
extraction finished success


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [22]:
# data preparation
df = pd.read_csv(CSV_PATH)

# get cleaned dataframe (make sure 'df' is defined from your csv loading cell)
df_utterance = df[df["KEY"].str.endswith("_u")].copy()
df_utterance["label"] = df_utterance["Sarcasm"].astype(int)

# split data
train_df, temp_df = train_test_split(df_utterance, test_size=0.3, random_state=42, stratify=df_utterance["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])


In [ ]:
def precalculate_whisper_features(df, stats_json, audio_dir, feature_extractor, save_dir):
    """pre-calculate mel spectrograms and save them to disk as tensors"""
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    with open(stats_json, 'r') as f:
        timestamps = {item['key']: item for item in json.load(f)}

    print("starting feature extraction...")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        key = row['KEY']
        save_path = os.path.join(save_dir, f"{key}.pt")

        if os.path.exists(save_path): continue

        # load and mono conversion
        waveform, sr = torchaudio.load(os.path.join(audio_dir, f"{key}.wav"))
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # resample once
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

        # crop
        ts = timestamps[key]
        start_idx = int(ts['start_of_speech'] * 16000)
        end_idx = int(ts['end_of_speech'] * 16000)
        waveform = waveform[:, start_idx:end_idx]

        # features
        mel = feature_extractor(
            waveform.squeeze(0).numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.squeeze(0)

        # save tensor
        torch.save(mel, save_path)


In [ ]:
FEATURE_DIR = "whisper_features_cache"
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
precalculate_whisper_features(df_utterance, STATS_JSON, AUDIO_DIR, feature_extractor, FEATURE_DIR)

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



preprocessor_config.json: 0.00B [00:00, ?B/s]

starting feature extraction...


  0%|          | 0/1201 [00:00<?, ?it/s]

In [23]:
FEATURE_DIR = "whisper_features_cache"

In [ ]:
# start training
# the model will be stored in 'trained_model' variable for future use
audio_report, trained_whisper_model = train_and_evaluate(
    train_df,
    val_df,
    test_df,
    STATS_JSON,
    AUDIO_DIR,
    cached=True,
    feature_dir=FEATURE_DIR
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Processing epoch:   0%|          | 0/30 [00:00<?, ?it/s]

Processing batches:   0%|          | 0/105 [00:00<?, ?it/s]

epoch 0 validation fone score 0.3333


Processing batches:   0%|          | 0/105 [00:00<?, ?it/s]

...

epoch 27 validation fone score 0.5667


Processing batches:   0%|          | 0/105 [00:00<?, ?it/s]

epoch 28 validation fone score 0.5884


Processing batches:   0%|          | 0/105 [00:00<?, ?it/s]

epoch 29 validation fone score 0.5658

final research report
{
    "test_accuracy": 0.5580110497237569,
    "test_f1_macro": 0.551980198019802,
    "inference_latency_ms": 11.598420209975302,
    "latency_std_ms": 3.2301680556179626,
    "total_flops": 130729356288.0,
    "total_gflops": 130.729356288,
    "model_parameters_count": 87399426.0,
    "peak_vram_mb": 0.0
}


## Save Whisper Embeddings (npy)

In [24]:
import torch
import numpy as np
import os
import json
from torch.utils.data import DataLoader
from transformers import WhisperModel


def extract_whisper_embeddings_all(train_df, val_df, test_df, feature_dir, model_path):
    """load whisper classifier and extract embeddings for track three fusion"""

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # initialize backbone and architecture
    # we use the encoder from whisper small
    whisper_model = WhisperModel.from_pretrained("openai/whisper-small")
    whisper_encoder = whisper_model.get_encoder()

    # recreate the model structure used during whisper training
    model = RobustAttentiveClassifier(whisper_encoder, hidden_size=768)

    # load the best saved weights
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()
    print(f"whisper model loaded from {model_path}")

    # 3 define data splits to process
    splits = {
        "train": train_df,
        "val": val_df,
        "test": test_df
    }

    whisper_embeddings = {}

    # processing loop
    with torch.no_grad():
        for name, df in splits.items():
            print(f"processing {name} split for whisper features")
            # use the cached dataset pointing to whisper spectrograms
            dataset = CachedMustardDataset(df, feature_dir, augment=False)
            # no collate_fn needed for whisper as tensors are fixed size
            loader = DataLoader(dataset, batch_size=16, shuffle=False)

            for batch in loader:
                # whisper features are [batch, 80, 3000]
                x = batch["input_features"].to(device)
                keys = batch["key"]

                # forward pass to get the attentive pooled vector
                _, pooled = model(x)

                # convert tensor to numpy and map to key
                pooled_np = pooled.cpu().numpy()
                for i, key in enumerate(keys):
                    whisper_embeddings[key] = pooled_np[i]

    # save the complete dictionary for multimodal experiments
    save_path = "audio_embeddings_whisper_full.npy"
    np.save(save_path, whisper_embeddings)
    print(f"success saved {len(whisper_embeddings)} embeddings to {save_path}")
    return save_path

WHISPER_CACHE = "whisper_features_cache"
WHISPER_WEIGHTS = "/content/Whisper/best_audio_model.pt"

extract_whisper_embeddings_all(train_df, val_df, test_df, WHISPER_CACHE, WHISPER_WEIGHTS)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

whisper model loaded from /content/Whisper/best_audio_model.pt
processing train split for whisper features
processing val split for whisper features
processing test split for whisper features
success saved 1201 embeddings to audio_embeddings_whisper_full.npy


'audio_embeddings_whisper_full.npy'

# WavLM

In [17]:
def audio_collate_fn(batch):
    """pad variable length sequences to the max length in a batch"""
    features = [item["input_features"] for item in batch]
    labels = [item["label"] for item in batch]
    keys = [item["key"] for item in batch]

    # set absolute max length (e.g., 10 seconds = 160,000 samples)
    # this prevents extreme padding issues
    max_len = 160000
    features = [f[:max_len] if f.size(0) > max_len else f for f in features]

    padded_features = torch.nn.utils.rnn.pad_sequence(features, batch_first=True)

    # create attention mask (1 for real data, 0 for padding)
    # this is critical for wavlm transformer blocks
    masks = [torch.ones(f.size(0)) for f in features]
    padded_masks = torch.nn.utils.rnn.pad_sequence(
        masks,
        batch_first=True,
        padding_value=0.0
    )

    return {
        "input_features": padded_features,
        "attention_mask": padded_masks,
        "label": torch.stack(labels),
        "key": keys
    }

In [18]:
class RobustAttentiveClassifier(nn.Module):
    """audio classifier with multihead attention pooling and normalization"""

    def __init__(self, backbone, hidden_size, num_heads=4):
        """setup architecture with frozen encoder and trainable head"""
        super().__init__()
        self.backbone = backbone
        # freeze encoder parameters to save resources
        for param in self.backbone.parameters():
            param.requires_grad = False

        self.layer_norm = nn.LayerNorm(hidden_size)
        self.mha = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
        # learnable query for pooling axis
        self.query = nn.Parameter(torch.randn(1, 1, hidden_size))

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 2)
        )

    def forward(self, input_features, attention_mask=None):
        """process features with memory optimization"""

        # CRITICAL: Disable gradient tracking for the frozen backbone
        # This prevents storing activations and saves massive VRAM
        with torch.no_grad():
            outputs = self.backbone(input_features, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state # [batch, time, hidden]

        # apply normalization and attention (these will have gradients)
        hidden_states = self.layer_norm(hidden_states)

        batch_size = hidden_states.size(0)
        query = self.query.expand(batch_size, -1, -1)

        attn_out, _ = self.mha(query, hidden_states, hidden_states)

        pooled = attn_out.squeeze(1)
        logits = self.head(pooled)
        return logits, pooled


In [19]:
class CachedMustardDataset(Dataset):
    """lightning fast dataset with safety checks for empty or short tensors"""

    def __init__(self, data_frame, feature_dir, *args, augment=False, **kwargs):
        """initialize and set paths"""
        self.df = data_frame
        self.feature_dir = feature_dir
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def apply_adaptive_augment(self, features):
        """apply augmentation with safety checks for zero length sequences"""
        # check if features are empty to avoid reshape errors
        if features.numel() == 0:
            return features

        if len(features.shape) == 2:
            # whisper mel spectrograms [80, time]
            # safety check: time dimension must be larger than mask parameters
            if features.size(1) <= 35:
                return features

            freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
            time_mask = torchaudio.transforms.TimeMasking(time_mask_param=35)
            return time_mask(freq_mask(features))

        elif len(features.shape) == 1:
            # wavlm raw values [time]
            # safety check: time dimension must be larger than mask parameter
            if features.size(0) <= 50:
                return features

            features = features.unsqueeze(0)
            time_mask = torchaudio.transforms.TimeMasking(time_mask_param=50)
            augmented = time_mask(features)
            noise = torch.randn_like(augmented) * 0.001
            return (augmented + noise).squeeze(0)

        return features

    def __getitem__(self, idx):
        """load tensor and ensure it is not empty before processing"""
        row = self.df.iloc[idx]
        key = row['KEY']

        feature_path = os.path.join(self.feature_dir, f"{key}.pt")

        try:
            features = torch.load(feature_path)
        except Exception:
            # fallback for missing or corrupted files
            features = torch.zeros((16000,)) # one second of silence

        # global safety check: if tensor is empty, replace with tiny silence
        if features.numel() == 0:
            # use 1600 samples (0.1s) as a safe minimum for the model
            features = torch.zeros((1600,)) if len(features.shape) == 1 else torch.zeros((80, 100))

        if self.augment:
            features = self.apply_adaptive_augment(features)

        return {
            "input_features": features,
            "label": torch.tensor(row['label'], dtype=torch.long),
            "key": key
        }

In [20]:
class WavLMFeaturePrecalculator:
    """utility to extract and cache wavlm features to disk"""

    def __init__(self, model_id, stats_json, audio_dir, save_dir):
        """initialize paths and feature extractor"""
        self.extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)
        self.audio_dir = audio_dir
        self.save_dir = save_dir
        with open(stats_json, 'r') as f:
            self.timestamps = {item['key']: item for item in json.load(f)}

        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

    def run(self, df):
        """process all audio files and save as pytorch tensors"""
        print("starting wavlm feature extraction")
        for _, row in tqdm(df.iterrows(), total=len(df)):
            key = row['KEY']
            save_path = os.path.join(self.save_dir, f"{key}.pt")
            if os.path.exists(save_path):
                continue

            # load and convert to mono
            waveform, sr = torchaudio.load(os.path.join(self.audio_dir, f"{key}.wav"))
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            # handle resampling
            if sr != 16000:
                waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

            # crop using whisper timestamps for alignment
            ts = self.timestamps[key]
            start_idx = int(ts['start_of_speech'] * 16000)
            end_idx = int(ts['end_of_speech'] * 16000)
            waveform = waveform[:, start_idx:end_idx]

            # wavlm expects input values instead of mel features
            # normalization is handled by the extractor
            inputs = self.extractor(
                waveform.squeeze(0).numpy(),
                sampling_rate=16000,
                return_tensors="pt"
            ).input_values.squeeze(0)

            torch.save(inputs, save_path)


In [21]:
import torch.nn.functional as f

class WavLMRobustClassifier(nn.Module):
    """advanced wavlm classifier with learnable layer weights and memory optimization"""

    def __init__(self, backbone, hidden_size):
        """initialize model with weighted layer pooling and deep mlp head"""
        super().__init__()
        self.backbone = backbone
        # freeze the entire backbone to prevent overfitting on small data
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.num_layers = 13 # input plus twelve blocks
        self.layer_weights = nn.Parameter(torch.ones(self.num_layers))
        self.layer_norm = nn.LayerNorm(hidden_size)

        # deep head with aggressive dropout to fight overfitting
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 2)
        )

    def forward(self, input_features, attention_mask=None):
        """extract all layers and perform weighted sum with downsampled mask"""

        with torch.no_grad():
            # fetch all thirteen hidden states from frozen backbone
            outputs = self.backbone(
                input_features,
                attention_mask=attention_mask,
                output_hidden_states=True
            )

        # compute layer weights via softmax
        weights = f.softmax(self.layer_weights, dim=0)

        # memory efficient weighted sum of hidden states
        # avoid using torch stack to save ram
        weighted_states = 0
        for i in range(self.num_layers):
            weighted_states += weights[i] * outputs.hidden_states[i]

        # apply pooling over time axis
        if attention_mask is not None:
            # downsample attention mask to match hidden states sequence length
            # wavlm compresses time dimension by factor of three hundred twenty
            # we use nearest interpolation to keep mask binary
            downsampled_mask = f.interpolate(
                attention_mask.unsqueeze(1),
                size=weighted_states.size(1),
                mode='nearest'
            ).squeeze(1)

            # apply mask and calculate mean only for valid speech segments
            masked_states = weighted_states * downsampled_mask.unsqueeze(-1)
            sum_states = masked_states.sum(dim=1)
            count_states = downsampled_mask.sum(dim=1).unsqueeze(-1).clamp(min=1e-9)
            pooled = sum_states / count_states
        else:
            # fallback to simple mean if mask is not provided
            pooled = weighted_states.mean(dim=1)

        pooled = self.layer_norm(pooled)
        logits = self.head(pooled)

        return logits, pooled

In [22]:
from transformers import AutoModel, Wav2Vec2FeatureExtractor

def train_and_evaluate_robust(train_df, val_df, test_df, feature_dir):
    """enhanced training loop with scheduler and history tracking"""

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # backbone initialization
    model_id = "microsoft/wavlm-base"
    backbone = AutoModel.from_pretrained(model_id)

    # data loaders using the cached dataset class
    train_ds = CachedMustardDataset(train_df, feature_dir, augment=True)
    val_ds = CachedMustardDataset(val_df, feature_dir, augment=False)
    test_ds = CachedMustardDataset(test_df, feature_dir, augment=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=8,
        shuffle=True,
        num_workers=0,
        collate_fn=audio_collate_fn
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=8,
        shuffle=False,
        num_workers=0,
        collate_fn=audio_collate_fn
    )
    # for test we also use it to maintain consistency
    test_loader = DataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=audio_collate_fn
    )

    # model with multihead attention
    model = WavLMRobustClassifier(backbone, hidden_size=768)
    model.to(device)

    # optimization setup with reduced learning rate
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    # add scheduler to reduce lr on plateau
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=3
    )

    # tracking history for analysis and plotting
    history = {
        "train_loss": [],
        "val_f1": []
    }

    best_val_f1 = 0.0

    for epoch in tqdm(range(30), position=0, leave=True, desc="Process epochs"):
        model.train()
        running_loss = 0.0
        for batch in tqdm(train_loader, position=1, leave=True, desc="Process batches"):
            x = batch["input_features"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["label"].to(device)

            optimizer.zero_grad()
            logits, _ = model(x, attention_mask=mask)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)

        # validation phase
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                x = batch["input_features"].to(device)
                mask = batch["attention_mask"].to(device)
                y = batch["label"].to(device)
                logits, _ = model(x, attention_mask=mask)
                preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                targets.extend(y.cpu().numpy())

        current_f1 = f1_score(targets, preds, average='macro')

        # record history
        history["train_loss"].append(avg_train_loss)
        history["val_f1"].append(current_f1)

        # update scheduler based on validation performance
        scheduler.step(current_f1)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"epoch {epoch} | loss {avg_train_loss:.4f} | val f1 {current_f1:.4f} | lr {current_lr:.6f}")

        if current_f1 > best_val_f1:
            best_val_f1 = current_f1
            torch.save(model.state_dict(), "best_wavlm_model.pt")

        # final performance evaluation
    model.load_state_dict(torch.load("best_wavlm_model.pt"))
    model.eval()

    profiler = PerformanceProfiler(device)
    test_preds, test_labels, embeddings_dict = [], [], {}

    # post training feature extraction for next fusion models
    model.eval()
    embeddings_dict = {}
    with torch.no_grad():
        for batch in test_loader:
            x = batch["input_features"].to(device)
            mask = batch["attention_mask"].to(device)
            y = batch["label"].to(device)
            key = batch["key"][0]

            logits, pooled = model(x, attention_mask=mask)
            test_preds.append(torch.argmax(logits, dim=1).item())
            test_labels.append(y.item())
            # save for track three
            embeddings_dict[key] = pooled.cpu().numpy()

    np.save("audio_embeddings_wavlm.npy", embeddings_dict)

    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds, average='macro')
    sample_batch = next(iter(test_loader))
    flops, params = profiler.measure_flops(model, sample_batch["input_features"])
    mean_lat, std_lat = profiler.measure_latency(model, test_loader)

    # computational metrics
    sample_batch = next(iter(test_loader))["input_features"]
    flops, params = profiler.measure_flops(model, sample_batch)
    mean_lat, std_lat = profiler.measure_latency(model, test_loader)
    vram = profiler.get_memory_stats()

    # aggregate all research data
    final_report = {
        "test_accuracy": acc,
        "test_f1_macro": f1,
        "inference_latency_ms": mean_lat,
        "latency_std_ms": std_lat,
        "total_flops": flops,
        "total_gflops": flops / 1e9,
        "model_parameters_count": params,
        "peak_vram_mb": vram
    }

    print("\nfinal research report")
    print(json.dumps(final_report, indent=4))

    # save embeddings for multimodal track
    with open("audio_embeddings.json", "w") as f:
        json.dump(embeddings_dict, f)

    return history, final_report, model

## Preprocessing Audio (mel-spectrograms)

In [ ]:
# clear memory from previous whisper run
if 'trained_whisper_model' in locals():
    del trained_whisper_model
gc.collect()
torch.cuda.empty_cache()

# configuration for wavlm
WAVLM_MODEL_ID = "microsoft/wavlm-base"
WAVLM_CACHE_DIR = "wavlm_features_cache"

# initialize wavlm feature extractor
# unlike whisper it produces normalized raw waveforms
wavlm_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_MODEL_ID)

# step one: precalculate features for wavlm
# this replaces mel spectrograms with wavlm input values
precalculator = WavLMFeaturePrecalculator(
    model_id=WAVLM_MODEL_ID,
    stats_json=STATS_JSON,
    audio_dir=AUDIO_DIR,
    save_dir=WAVLM_CACHE_DIR
)
precalculator.run(df_utterance)

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

starting wavlm feature extraction


  0%|          | 0/1201 [00:00<?, ?it/s]

## Train and evaluate

In [23]:
if 'trained_whisper_model' in locals(): del trained_whisper_model
if 'model' in locals(): del model

# clear cache and collect garbage
gc.collect()
torch.cuda.empty_cache()

# confirm clean state
print(f"vram allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} mb")

vram allocated: 0.00 mb


In [ ]:
history_wavlm, report_wavlm, trained_wavlm_model = train_and_evaluate_robust(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_dir=WAVLM_CACHE_DIR
)

# save the final report
with open("wavlm_track_report.json", "w") as f:
    json.dump(report_wavlm, f, indent=4)

# save history for plotting later
with open("wavlm_training_history.json", "w") as f:
    json.dump(history_wavlm, f, indent=4)

print("wavlm pipeline finished successfully")

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Process epochs:   0%|          | 0/30 [00:00<?, ?it/s]

Process batches:   0%|          | 0/105 [00:00<?, ?it/s]

  key_padding_mask = _canonical_mask(



epoch 0 | loss 0.6765 | val f1 0.5369 | lr 0.000100


...

  key_padding_mask = _canonical_mask(



epoch 29 | loss 0.3082 | val f1 0.5915 | lr 0.000006


  key_padding_mask = _canonical_mask(




final research report
{
    "test_accuracy": 0.6464088397790055,
    "test_f1_macro": 0.6432618871643262,
    "inference_latency_ms": 24.774309609947522,
    "latency_std_ms": 4.367217556427591,
    "total_flops": 13645806848.0,
    "total_gflops": 13.645806848,
    "model_parameters_count": 61540322.0,
    "peak_vram_mb": 0.0
}


TypeError: Object of type ndarray is not JSON serializable

## Save WavLM Embeddings (npy)

In [25]:
def extract_all_embeddings(train_df, val_df, test_df, feature_dir, model_path):
    """load the best model and save embeddings for all splits to one npy file"""

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # initialize architecture
    model_id = "microsoft/wavlm-base"
    backbone = AutoModel.from_pretrained(model_id)
    model = WavLMRobustClassifier(backbone, hidden_size=768)

    # load weights
    model.load_state_dict(torch.load(model_path), strict=False)
    model.to(device)
    model.eval()
    print(f"Model loaded from {model_path}")

    # prepare loaders for all splits (batch_size can be larger here to speed up)
    splits = {
        "train": train_df,
        "val": val_df,
        "test": test_df
    }

    all_embeddings = {}

    with torch.no_grad():
        for name, df in splits.items():
            print(f"processing {name} split...")
            dataset = CachedMustardDataset(df, feature_dir, augment=False)
            loader = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=audio_collate_fn)

            for batch in loader:
                x = batch["input_features"].to(device)
                mask = batch.get("attention_mask").to(device)
                keys = batch["key"] # list of keys in the batch

                # get pooled output [batch, 768]
                _, pooled = model(x, attention_mask=mask)

                # convert to numpy and store by key
                pooled_np = pooled.cpu().numpy()
                for i, key in enumerate(keys):
                    all_embeddings[key] = pooled_np[i]

    # save as a single dictionary object for anya
    save_path = "audio_embeddings_wavlm_full.npy"
    np.save(save_path, all_embeddings)
    print(f"done! saved {len(all_embeddings)} embeddings to {save_path}")
    return save_path

CACHE_DIR = "/content/wavlm_features_cache"
MODEL_FILE = "/content/WavLM2/best_wavlm_model.pt"

extract_all_embeddings(train_df, val_df, test_df, CACHE_DIR, MODEL_FILE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Model loaded from /content/WavLM2/best_wavlm_model.pt
processing train split...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


processing val split...
processing test split...
done! saved 1201 embeddings to audio_embeddings_wavlm_full.npy


'audio_embeddings_wavlm_full.npy'